In [1]:
#seperate notebook for training the model
import os
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(0)
device = torch.device("cpu") #bc i'm doing this in aws, no gpu needed

transform = transforms.ToTensor() #this is a torchvision package for casting images to torch.Tensor

train_ds = datasets.MNIST(root="", train=True, download=True, transform=transform) #train=True just signals which part of MNIST to download
test_ds  = datasets.MNIST(root="", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True) #shuffle is needed to stop model from learning patterns in file order
test_loader = DataLoader(test_ds, batch_size=56, shuffle=False)

model = nn.Sequential(
    nn.Flatten(), #unsurprisingly, this flattens data 0
    nn.Linear(28*28, 256), #linear layer 1
    nn.ReLU(), # 2
    nn.Linear(256, 128), #squash down 3
    nn.ReLU(), # 4
    nn.Linear(128, 10) #output 5
).to(device)

loss_fn = nn.CrossEntropyLoss() #this expects logits
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 5

for epoch in range(1, epochs + 1): #loop
    model.train() #activates things like dropout

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for x, y in train_loader: #for each batch
        x = x.to(device)
        y = y.to(device)

        logits = model(x)#whatever
        loss = loss_fn(logits, y)
        #print(logits)

        optimizer.zero_grad(set_to_none=True)#reset optimizer from last time around
        loss.backward()#calculate loss backward
        optimizer.step()#and reset stuff

        train_loss_sum += loss.item() * x.size(0) #.item() extracts a python value from a single-element tensor
        preds = logits.argmax(dim=1) #pick the index of the largest logit
        train_correct += (preds == y).sum().item()
        train_total += x.size(0)

    train_loss = train_loss_sum / train_total 
    train_acc = train_correct / train_total 

    model.eval()
    test_loss_sum = 0.0
    test_correct = 0
    test_total = 0

    with torch.no_grad():#batch eval
        for x, y in test_loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = loss_fn(logits, y)

            test_loss_sum += loss.item() * x.size(0)
            preds = logits.argmax(dim=1)
            test_correct += (preds == y).sum().item()
            test_total += x.size(0)

    test_loss = test_loss_sum / test_total
    test_acc = test_correct / test_total

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_loss:.4f}, train acc {train_acc*100:.2f}% | "
        f"test loss {test_loss:.4f}, test acc {test_acc*100:.2f}%"
    )

torch.save(model.state_dict(), os.path.join("", "writeup_mlp_mnist_state_dict.pt"))

100%|██████████| 9.91M/9.91M [00:00<00:00, 25.4MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 43.6MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 190MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.21MB/s]


Epoch 01 | train loss 0.3592, train acc 89.88% | test loss 0.1695, test acc 94.79%
Epoch 02 | train loss 0.1410, train acc 95.81% | test loss 0.1090, test acc 96.61%
Epoch 03 | train loss 0.0924, train acc 97.19% | test loss 0.0929, test acc 97.08%
Epoch 04 | train loss 0.0674, train acc 97.91% | test loss 0.0834, test acc 97.48%
Epoch 05 | train loss 0.0507, train acc 98.44% | test loss 0.0823, test acc 97.41%
